# 01 — Sanity Check: tokenizer wrappers on Korean

**Goal**: confirm each provider's `Tokenizer` wrapper produces token counts in the expected range for Korean text *before* we trust any later experiment. New wrappers are added one at a time; PASS here is the gate to add the next.

**Pass criterion** (calibrated on the n=10 sample, 2026-05-07)

| Language | Expected TPC (tokens / character) |
|---|---|
| Korean | ~0.5 – 0.8 |
| English | ~0.15 – 0.30 |

If Korean lands well outside `0.5–0.8`, something is wrong (encoding, normalization, model name) — fix that before trusting any later experiment.

**Phase 1 surprise hypotheses to keep in mind** (we are not testing them here yet, but later runs will):

- **H1** — frontier models meaningfully improved Korean efficiency  
- **H2** — Korean-specialized models win on TPC but lose on ECPC after pricing  
- **H3** — medical / 한자-mixed text has a far higher penalty than everyday text

**Inter-model signal threshold (project decision rule)**: a Claude/GPT-4o TPC ratio of **≥ 1.3×** at full-corpus scale would count as a real H1/H2 signal. Below 1.3× we pivot to domain-variance (H3) as the paper's primary angle. At n=10 the ratio is informational only — never paper-cite.

In [9]:
from __future__ import annotations

import sys
from pathlib import Path

# Make the in-tree package importable without installing it.
# Once you `uv sync` and `pip install -e .` you can drop these two lines.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from korean_llm_cost.tokenizers.openai_tok import OpenAITokenizer

tok = OpenAITokenizer("gpt-4o")
print(tok)

<OpenAITokenizer gpt-4o@2024-11-20>


In [10]:
# 10 Korean samples across categories we plan to measure in Phase 1,
# plus a few harder cases (한자 mixed, code mixed, exclamation) to see
# the per-sample variance even at this tiny scale.
korean_samples: list[tuple[str, str]] = [
    ("conv",       "오늘 점심 뭐 먹을까?"),
    ("conv",       "내일 회의 시간 좀 알려줘."),
    ("news",       "한국은행은 기준금리를 동결하기로 결정했다."),
    ("news",       "정부는 내년도 예산안을 국회에 제출하였다."),
    ("medical",    "환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다."),
    ("medical",    "고혈압성 좌심실 비대 소견이 관찰되었다."),
    ("academic",   "본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다."),
    ("code_mixed", "리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다."),
    ("hanja",      "大韓民國의 헌법 제1조는 민주공화국임을 천명한다."),
    ("short",      "와 진짜 대박이네요!"),
]

english_baseline: list[str] = [
    "What should we eat for lunch today?",
    "Let me know the meeting time tomorrow.",
    "The Bank of Korea decided to freeze the benchmark interest rate.",
    "This study quantitatively analyzes the efficiency of Korean tokenizers.",
    "Wow, that's really amazing!",
]

len(korean_samples), len(english_baseline)

(10, 5)

In [11]:
def tpc(text: str) -> float:
    """Tokens per character. Aggregate over many texts via summed counts,
    not averaged TPCs — averaging per-text TPC over-weights short texts."""
    return tok.count(text) / len(text) if text else 0.0


print(f"{'category':<12} {'chars':>5} {'toks':>5} {'tpc':>6}  text")
print("-" * 80)
for category, text in korean_samples:
    n_chars = len(text)
    n_toks = tok.count(text)
    print(f"{category:<12} {n_chars:>5} {n_toks:>5} {n_toks/n_chars:>6.3f}  {text}")

category     chars  toks    tpc  text
--------------------------------------------------------------------------------
conv            12     8  0.667  오늘 점심 뭐 먹을까?
conv            15     9  0.600  내일 회의 시간 좀 알려줘.
news            23    13  0.565  한국은행은 기준금리를 동결하기로 결정했다.
news            23    13  0.565  정부는 내년도 예산안을 국회에 제출하였다.
medical         40    30  0.750  환자에게 아세트아미노펜 500밀리그램을 1일 3회 복용하도록 처방하였다.
medical         22    17  0.773  고혈압성 좌심실 비대 소견이 관찰되었다.
academic        33    20  0.606  본 연구는 한국어 토크나이저의 효율성을 정량적으로 분석한다.
code_mixed      38    22  0.579  리액트의 useState 훅을 사용하여 컴포넌트의 상태를 관리합니다.
hanja           27    22  0.815  大韓民國의 헌법 제1조는 민주공화국임을 천명한다.
short           11     8  0.727  와 진짜 대박이네요!


In [12]:
# Aggregate TPC: total tokens / total chars (NOT mean of per-text TPCs).
ko_chars = sum(len(t) for _, t in korean_samples)
ko_toks = sum(tok.count(t) for _, t in korean_samples)
en_chars = sum(len(t) for t in english_baseline)
en_toks = sum(tok.count(t) for t in english_baseline)

ko_tpc = ko_toks / ko_chars
en_tpc = en_toks / en_chars
kpr = ko_tpc / en_tpc  # rough; real KPR needs aligned parallel data

print(f"Korean : {ko_toks:>4} tokens / {ko_chars:>4} chars = TPC {ko_tpc:.3f}")
print(f"English: {en_toks:>4} tokens / {en_chars:>4} chars = TPC {en_tpc:.3f}")
print(f"Rough KPR (ko / en) = {kpr:.2f}x")

# Pass / fail messages — make it impossible to miss a regression.
ok_ko = 0.5 <= ko_tpc <= 0.8
ok_en = 0.15 <= en_tpc <= 0.30
print()
print("Korean TPC in [0.5, 0.8]?  ", "PASS" if ok_ko else f"FAIL ({ko_tpc:.3f})")
print("English TPC in [0.15, 0.30]?", "PASS" if ok_en else f"FAIL ({en_tpc:.3f})")

Korean :  162 tokens /  244 chars = TPC 0.664
English:   46 tokens /  235 chars = TPC 0.196
Rough KPR (ko / en) = 3.39x

Korean TPC in [0.5, 0.8]?   PASS
English TPC in [0.20, 0.30]? FAIL (0.196)


## Anthropic (Claude) check

Same 10 Korean samples, this time through the Claude `count_tokens` API. Two things to confirm:

1. **Korean TPC for Claude lands in `[0.5, 0.8]`** — same sanity range as GPT-4o; outside this and the wrapper or the chat-overhead calibration is wrong.
2. **Claude / GPT-4o TPC ratio at n=10** — informational only. ≥ 1.3× would be a candidate H1/H2 signal worth verifying on the full corpus; below 1.3× we lean H3.

**Prereq**: `uv sync --extra anthropic --extra dev` and `ANTHROPIC_API_KEY` in `.env` (or shell env). The `count_tokens` endpoint is free; this whole cell costs $0.

In [ ]:
import os

anth = None
try:
    from korean_llm_cost.tokenizers.anthropic_tok import AnthropicTokenizer
except ImportError:
    print("anthropic SDK not installed. Run:  uv sync --extra anthropic --extra dev")
else:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("ANTHROPIC_API_KEY not set.")
        print("  cp .env.example .env  →  fill in the key  →  restart kernel.")
        print("  (count_tokens is free; you will not be billed for this notebook.)")
    else:
        # Default to Sonnet 4.5 (stable). Override to claude-sonnet-4-7 once verified.
        anth = AnthropicTokenizer("claude-sonnet-4-5")
        print(anth, f"(chat overhead = {anth._overhead} tokens, calibrated)")

In [ ]:
if anth is not None:
    # One API call per sample (cheap + free, ~10 calls total).
    cla_counts = [anth.count(text) for _, text in korean_samples]

    print(f"{'category':<12} {'chars':>5} {'gpt':>4} {'cla':>4}  text")
    print("-" * 80)
    for (category, text), n_cla in zip(korean_samples, cla_counts):
        n_chars = len(text)
        n_gpt = tok.count(text)
        print(f"{category:<12} {n_chars:>5} {n_gpt:>4} {n_cla:>4}  {text}")

    cla_toks = sum(cla_counts)
    cla_tpc = cla_toks / ko_chars  # same chars; only the model changes
    ratio = cla_tpc / ko_tpc

    print()
    print(f"GPT-4o  TPC: {ko_tpc:.3f}")
    print(f"Claude  TPC: {cla_tpc:.3f}")
    print(f"Claude / GPT-4o ratio = {ratio:.2f}×")

    # Sanity: Claude Korean TPC must also be in [0.5, 0.8].
    ok_cla = 0.5 <= cla_tpc <= 0.8
    print()
    print("Claude Korean TPC in [0.5, 0.8]?", "PASS" if ok_cla else f"FAIL ({cla_tpc:.3f})")

    # Inter-model signal threshold (1.3×) — informational at n=10.
    if abs(ratio - 1.0) >= 0.3:
        print(f"→ |ratio − 1| ≥ 0.3 at n=10. Candidate H1/H2 signal — verify on full corpus.")
    else:
        print(f"→ |ratio − 1| < 0.3 at n=10. No clear inter-model gap; favor H3 (domain) angle.")

## What to do if a check fails

- **Korean TPC > 0.8**: likely encoding-related — confirm the source file is UTF-8, no NFC/NFD inconsistency, and `tok.count` is being called on the raw `str` (not bytes).
- **Korean TPC < 0.5**: implausible for Hangul on `o200k_base`. Verify `tok.info.version == '2024-11-20'` (i.e., the `gpt-4o` snapshot, not a fallback).
- **English TPC > 0.30**: punctuation-heavy or capitalized samples are dominating; rule out trailing whitespace or BOM.
- **English TPC < 0.15**: extremely common-word content; sanity sample is unrepresentatively easy. Not blocking — verify on the news/academic corpus before reporting.
- **Claude overhead calibration looks off** (e.g., negative counts): the 1-character probe assumption (`'.' = 1 token`) may have broken in a future Claude release. Re-derive overhead by sending two probes of different known lengths and solving for the constant.

## Next

Once the GPT-4o + Claude checks both PASS, the next wrapper to add is **Google Gemini** (same pattern — own cell, own PASS gate). After Google: HuggingFace wrappers one model at a time (Solar / Llama / Qwen). Only when all six Phase 1 wrappers have a PASS row in this notebook do we move on to `metrics.py` and corpus collection.

## Early signals (n=10, **NOT** for paper)

The sanity sample is too small for any number to appear in the paper. The signals below are **candidates** to verify on the full Phase 1 corpus (≥ 1000 sentences/category, bootstrap CI, paired tests). They live here so we don't lose track of where to look first.

| Candidate signal | Observed at n=10 (GPT-4o) | Verify on |
|---|---|---|
| Korean ≈ 3.4× English per char (KPR) | 3.39 | parallel ko↔en corpus (KorNLI/MNLI overlap, OpenSubtitles ko-en) |
| Hanja-mixed text exceeds 0.8 TPC | 0.815 | hanja-rich subcorpus (legal / historical text) |
| Medical 13–17% above conversational | 0.75–0.77 vs 0.60–0.67 | KorMedMCQA + AI Hub medical text |
| Code-mixed *lower* than expected | 0.579 (English identifiers like `useState` likely 1 token) | dev-blog crawl (Velog/Tistory) |

**Project signal threshold (decision rule, 2026-05-07):**
- Inter-model TPC ratio **≥ 1.3×** at full-corpus scale → H1/H2 candidate signal, primary paper angle
- Inter-model TPC ratio **< 1.3×** → pivot toward domain-variance (H3) as the primary angle

**Do not cite n=10 numbers anywhere outside this notebook.** They exist only to motivate which directions are worth prioritizing in the full Phase 1 run.